In [ ]:
!pip install langchain-groq
!pip install langchain_community
!pip install Chromadb

In [ ]:
from annotated_types import GroupedMetadata
from google.colab import userdata
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    temperature=0,
    groq_api_key= userdata.get("GROQ_API_KEY"),
    model_name= "qwen/qwen3.8-27b"
)
response = llm.invoke("The first person to land on the moon was ...")
print(response.content)

The first person to land on the moon was **Neil Armstrong**.

He stepped onto the lunar surface on **July 20, 1969**, during the Apollo 11 mission, becoming the first human to set foot on the Moon. He was followed shortly after by Buzz Aldrin, while Michael Collins remained in orbit aboard the Command Module.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/director-software-engineering-core-cloud-itc/job/R-85259")
page_data = loader.load().pop().page_content
print (page_data)

/tmp/ipykernel_12598/3555358357.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader





















Director, Software Engineering Core Cloud, ITC














































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Retu

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is form the career's page of a website.
    Your job is to extract the job postings and return them in JSON format conta
    ining the following keys: `role`, `experience`, `skills` and `description`.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):
    """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data': page_data})
print(res.content)

{
  "role": "Director, Software Engineering Core Cloud, ITC",
  "experience": "15+ years leading engineering teams, including 3+ years at Director level managing managers or senior ICs; 5+ years of hands-on AWS engineering experience",
  "skills": [
    "Cloud governance",
    "Entitlements",
    "Policy systems",
    "IAM/RBAC",
    "Policy guardrails",
    "FinOps",
    "Cost attribution",
    "Cost optimization",
    "Terraform",
    "Crossplane",
    "CDK",
    "Distributed systems",
    "Kubernetes",
    "APIs",
    "Event-driven architecture",
    "Python",
    "JavaScript",
    "AI/Agentic capabilities",
    "Microservices",
    "Security-by-design",
    "Agile/DevOps",
    "CI/CD",
    "SLO-driven operations"
  ],
  "description": "Lead and grow 3–4 engineering squads, managing managers, principals, and senior engineers. Define technical direction and build a high-performance engineering culture within ITC. Drive multi-year platform roadmap and translate strategic into quarterl

In [ ]:
#The output above is still a string. To make it a real JSON object, the langchain
#parser is called

from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Director, Software Engineering Core Cloud, ITC',
 'experience': '15+ years leading engineering teams, including 3+ years at Director level managing managers or senior ICs; 5+ years of hands-on AWS engineering experience',
 'skills': ['Cloud governance',
  'Entitlements',
  'Policy systems',
  'IAM/RBAC',
  'Policy guardrails',
  'FinOps',
  'Cost attribution',
  'Cost optimization',
  'Terraform',
  'Crossplane',
  'CDK',
  'Distributed systems',
  'Kubernetes',
  'APIs',
  'Event-driven architecture',
  'Python',
  'JavaScript',
  'AI/Agentic capabilities',
  'Microservices',
  'Security-by-design',
  'Agile/DevOps',
  'CI/CD',
  'SLO-driven operations'],
 'description': 'Lead and grow 3–4 engineering squads, managing managers, principals, and senior engineers. Define technical direction and build a high-performance engineering culture within ITC. Drive multi-year platform roadmap and translate strategic into quarterly executions and ORKs. Own the cloud governance system inc

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/cold_email_generator_tool_Gen_AI_Llama_3/my_portfolio.csv')
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [ ]:
import chromadb
import uuid

client = chromadb.PersistentClient('/content/drive/MyDrive/Colab Notebooks/cold_email_generator_tool_Gen_AI_Llama_3/vectorstore')
collection = client.get_or_create_collection('my_portfolio')

if not collection.count():
  for _, row in df.iterrows():
    collection.add(documents=row["Techstack"],
                   metadatas={"links": row["Links"]},
                   ids=[str(uuid.uuid4())])

In [ ]:
links = collection.query(query_texts=["Experience in Python", "Expertise in React"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/react-portfolio'},
  {'links': 'https://example.com/react-native-portfolio'}]]

In [ ]:
job = json_res
job['skills']


['Cloud governance',
 'Entitlements',
 'Policy systems',
 'IAM/RBAC',
 'Policy guardrails',
 'FinOps',
 'Cost attribution',
 'Cost optimization',
 'Terraform',
 'Crossplane',
 'CDK',
 'Distributed systems',
 'Kubernetes',
 'APIs',
 'Event-driven architecture',
 'Python',
 'JavaScript',
 'AI/Agentic capabilities',
 'Microservices',
 'Security-by-design',
 'Agile/DevOps',
 'CI/CD',
 'SLO-driven operations']

In [ ]:
prompt_email = PromptTemplate.from_template(
    """
    ### JOB DESCRIPTION:
    {job_description}

    ### INSTRUCTION:
    You are Samuel, a senior software engineer at TheraPlan. TheraPlan is an AI
    & software company that enables the seamless integration of business process
    es through automated tools.
    Over our experience, we have empowered numerous enterprises with tailored
    solutions, fostering scalability, process optimization, cost reduction, and
    heightened overall efficiency.
    Your job is to write a cold email to the client regarding the job mentioned
    above describing the capability in fulfilling their needs.
    Also add the most relevant ones form the following links to showcase TheraPl
    an's portfolio: {link_list}
    Remember you are Samuel, SSE at TheraPlan.
    Do not Provide a preamble.
    ### EMAIL (NO PREAMBLE):
    """
)

chain_email = prompt_email | llm
res = chain_email.invoke(input={'job_description': str(job), 'link_list': links})
print(res.content)

Subject: Scaling ITC’s Core Cloud Platform: Governance, FinOps, and AI-Native Engineering

Hi [Hiring Manager's Name],

I’m writing to you from TheraPlan, where we specialize in enabling seamless business process integration through automated, scalable software solutions. Having reviewed the Director, Software Engineering Core Cloud, ITC role, I see a critical need for a leader who can bridge the gap between strategic cloud governance and hands-on technical execution—specifically in AWS, policy systems, and FinOps.

At TheraPlan, we have empowered numerous enterprises to achieve scalability and cost reduction by building robust, security-by-design platforms. Our approach aligns directly with your requirements for a high-performance engineering culture and a multi-year platform roadmap. We don’t just build software; we architect the governance and economic frameworks that allow engineering teams to scale without breaking the bank or compromising security.

Here is how our expertise maps

In [ ]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/


In [ ]:
!git config --global init.defaultBranch main

In [ ]:
!git branch -m main

In [ ]:
!git remote add origin https://github.com/Sammy-mercy/Langchain_Cold_email_generator

In [ ]:
!git add .

error: open("drive/MyDrive/Abdominal_Tuberculosis.gslides"): Operation not supported
error: unable to index file 'drive/MyDrive/Abdominal_Tuberculosis.gslides'
fatal: adding files failed


In [ ]:
%%writefile .gitignore
drive/
.config/
sample_data/
vectorstore/

### Track only the project directory in Git

To safely track only your specific notebook directory without running into Google Drive indexing issues, we will change directories to your project folder, initialize Git there, and run our Git workflow from that folder.

In [ ]:
import os

# 1. Define and navigate to the target project directory
project_dir = '/content/drive/MyDrive/Colab Notebooks/cold_email_generator_tool_Gen_AI_Llama_3'
os.chdir(project_dir)
print(f"Current working directory: {os.getcwd()}")

# 2. Initialize Git inside your project folder
!git init

# 3. Configure local repository parameters
!git config --global init.defaultBranch main
!git branch -m main

# 4. Add your GitHub remote URL
# (Make sure to remove any existing remote if you re-run this)
!git remote remove origin 2>/dev/null || true
!git remote add origin https://github.com/Sammy-mercy/Langchain_Cold_email_generator

# 5. Create a .gitignore specifically for your project folder
# This avoids tracking internal vector storage database files locally if you want to keep them off GitHub
with open('.gitignore', 'w') as f:
    f.write('vectorstore/\n__pycache__/\n.ipynb_checkpoints/\n')

# 6. Safely add and check status
!git add .
!git status

Once you have written the `.gitignore` file above to ignore the `drive/` mount, you can safely run `!git add .` again without indexing the Google Drive files.

In [1]:
import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/cold_email_generator_tool_Gen_AI_Llama_3')

In [2]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/drive/MyDrive/Colab Notebooks/cold_email_generator_tool_Gen_AI_Llama_3/.git/


In [3]:
!git config --global init.defaultBranch main

In [4]:
# 4. Add your GitHub remote URL
# (Make sure to remove any existing remote if you re-run this)
!git remote remove origin 2>/dev/null || true
!git remote add origin https://github.com/Sammy-mercy/Langchain_Cold_email_generator


In [5]:
 #5. Create a .gitignore specifically for your project folder
# This avoids tracking internal vector storage database files locally if you want to keep them off GitHub
with open('.gitignore', 'w') as f:
    f.write('vectorstore/\n__pycache__/\n.ipynb_checkpoints/\n')

SyntaxError: invalid syntax (2290470317.py, line 1)